In [ ]:
from torchvision import transforms
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image as PILImage
import numpy as np
import os
import pandas as pd
import random
from tqdm.auto import tqdm
import matplotlib as plt

In [ ]:
import quantus
from quantus import PointingGame, MaxSensitivity
import glob
from PIL import Image

from pytorch_grad_cam.utils.image import preprocess_image, show_cam_on_image
from quantus import norm_func, perturb_func, similarity_func
from datetime import datetime

from xai_lib import MemoryManagement as mm
from xai_lib import AttributionEngine as att_engine
import xaibenchmark as xb
from skimage.segmentation import slic

In [ ]:
name = "aveera"
PREPROCESSED_DIR = f"/data/{name}/preprocessed_idrid"
MODEL_NAMES = ["resnet50"]
N_TEST = 81
SEED=42

In [ ]:
_engine_cache = {}

def get_engine_for(model):
    """Look up the AttributionEngine that was built for this exact model object."""

    eng = _engine_cache.get(id(model))
    if eng is None:
        raise RuntimeError(
            "No AttributionEngine registered for this model."
        )
    return eng

In [ ]:
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Lets cuDNN pick the fastest conv algorithms since our input size (224x224) is fixed
torch.backends.cudnn.benchmark = True

In [ ]:
def get_predicted_labels(model, samples, device, transform=test_transform):
    """Recomputes the model's predicted class per sample."""
    preds = []
    model.eval()
    with torch.no_grad():
        for raw_img_np, _ in samples:
            input_tensor = transform(PILImage.fromarray(raw_img_np)).unsqueeze(0).to(device)
            pred = model(input_tensor).argmax(dim=1).item()
            preds.append(pred)
    return np.array(preds)

# **Explainable Artificial Intelligence**

**While Grad-CAM is widely applied in DR interpretability literature, disclosed hyperparameters for perturbation-based methods (LIME, SHAP) are rarely reported in DR-specific work.**

Our parameter choices were therefore informed by broader medical-imaging and XAI benchmarking literature rather than DR-specific research, while also accoutning for our resource constraints given the scale of our evaluation.

**For LIME, we set lime_samples=1000**, consistent with values used in [Gastrointestinal Endoscopic study](https://arxiv.org/pdf/2503.00780), [Sickle Cell study](https://arxiv.org/pdf/2305.01663) and [Photonic Chip study](https://arxiv.org/pdf/2505.09266) which set the number of samples to 1000 to ensure the creation of a sufficient number of perturbed samples around the input instance for accurate approximation.

**For SHAP, we set the max_evals =500**, consistent with the [Sickle Cell study](https://arxiv.org/pdf/2305.01663) from above and is also the SHAP libraries default. We also use batch_size-50 which is the SHAP library, the same justification as this study [haematopoietic stem cell study](https://pmc.ncbi.nlm.nih.gov/articles/PMC12746025/)


**We use a 32x32 blur kernel for the SHAP masker, smaller than the 128x128 kernel used in SHAP's reference examples for 224x224 natural images.** In natural-image classification, the relevant object typically occupies a large contiguous region of the frame whereas for DR lesion attribution, multiple small lesions can appear simultaneously across the image. A 128x128 kernel would blur across a majority of the image width in a single masking operation, conflating spatially distinct lesions that should be attributed independently, which is why we choose 32x32

**The XAI Attribution Engine is a single wrapper around three XAI methods, so each trained model can be explained the same way regardless of which method is used. Given a model and an image, it returns an attribution map per method highlighting the regions that most influenced the prediction**

**Loading the Refined IDRID dataset**

In [ ]:
def load_preprocessed_from_disk(save_dir):
    manifest_path = os.path.join(save_dir, "manifest.csv")
    if not os.path.exists(manifest_path):
        raise FileNotFoundError(
            f"No manifest.csv found in {save_dir}"
        )
    manifest = pd.read_csv(manifest_path).sort_values("index")

    samples = []
    for _, row in manifest.iterrows():
        img = np.array(Image.open(row["image_path"]).convert('RGB'))
        mask = np.array(Image.open(row["mask_path"]).convert('L'))
        samples.append((img, mask))
    return samples

In [ ]:
idrid_subset = load_preprocessed_from_disk(PREPROCESSED_DIR)

# **Side-by-Side Comparison of each XAI technique**

In [ ]:
raw_image, lesion_mask = idrid_subset[0]  # Sample image
drive_model_dir = f"/data/{name}/models"
FIGURES_DIR = f"/data/{name}/results/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

for model_name in MODEL_NAMES:
    drive_weight_path = os.path.join(drive_model_dir, f"{model_name}_best_model.pth")
    model = mm.load_model(model_name, drive_weight_path, device)

    engine = att_engine.AttributionEngine(model, device)
    _engine_cache[id(model)] = engine

    image_tensor = test_transform(PILImage.fromarray(raw_image))
    with torch.no_grad():
        predicted_class = model(image_tensor.unsqueeze(0).to(device)).argmax(dim=1).item()

    raw_img_uint8 = engine.unnormalize_to_uint8(image_tensor)
    img_batched = image_tensor.unsqueeze(0).to(device)

    print(f"--- {model_name} ---")

    engine.show_grad_cam(img_batched, raw_img_uint8, predicted_class)
    plt.savefig(os.path.join(FIGURES_DIR, f"gradcam_{model_name}.png"), dpi=300, bbox_inches="tight")
    plt.close('all')

    engine.show_shap(raw_img_uint8, target_class=predicted_class, all_classes=True)
    plt.savefig(os.path.join(FIGURES_DIR, f"shap_{model_name}.png"), dpi=300, bbox_inches="tight")
    plt.close('all')

    engine.show_lime(raw_img_uint8, target_class=predicted_class)
    plt.savefig(os.path.join(FIGURES_DIR, f"lime_{model_name}.png"), dpi=300, bbox_inches="tight")
    plt.close('all')

    mm.unload_model(model=model, engine=engine, engine_cache=_engine_cache)
    mm.gpu_mem_report(f"after unloading {model_name} (viz cell)")

    del image_tensor, img_batched, raw_img_uint8

    _engine_cache.clear()
    mm.gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()
    mm.gpu_mem_report(f"after unloading {model_name} (viz cell)")

In [ ]:
def _per_image_dir(save_dir, model_name):
    d = os.path.join(save_dir, f"{model_name}_per_image")
    os.makedirs(d, exist_ok=True)
    return d


def run_attributions_on_samples(engine, samples, transform, save_dir, model_name, desc="XAI"):
    per_image_dir = _per_image_dir(save_dir, model_name)

    for i, (raw_img, label) in enumerate(tqdm(samples, desc=desc, ncols=90)):
        img_path = os.path.join(per_image_dir, f"image_{i:04d}.npz")
        if os.path.exists(img_path):
            continue  

        img_tensor = transform(PILImage.fromarray(raw_img))
        try:
            maps = engine.compute_all(img_tensor, label)
            tmp_path = img_path + ".tmp"
            np.savez(
                tmp_path,
                lime=maps['lime'],
                shap=maps['shap'],
                gradcam=maps['gradcam'],
                label=label,
            )
            os.replace(tmp_path, img_path)
        finally:
            img_tensor = None
            mm.clear_memory()
            if i % 5 == 0:
                mm.log_rss(f"{desc}/image {i}")


def assemble_attribution_results(save_dir, model_name, n_images):
    per_image_dir = _per_image_dir(save_dir, model_name)
    lime_maps, shap_maps, gradcam_maps, labels = [], [], [], []
    for i in range(n_images):
        img_path = os.path.join(per_image_dir, f"image_{i:04d}.npz")
        with np.load(img_path) as d:
            lime_maps.append(d['lime'])
            shap_maps.append(d['shap'])
            gradcam_maps.append(d['gradcam'])
            labels.append(int(d['label']))
    return {
        'lime': lime_maps,
        'shap': shap_maps,
        'gradcam': gradcam_maps,
        'labels': labels,
    }

 **Computing attribution maps used for localization and faithfulness tests**

In [ ]:
N_TEST = 81
SAVE_DIR = f"/data/{name}/xai_maps"
os.makedirs(SAVE_DIR, exist_ok=True)

test_samples = idrid_subset[:N_TEST]

for model_name in MODEL_NAMES:
    save_path = os.path.join(SAVE_DIR, f"attribution_maps_{model_name}_test.npz")

    if os.path.exists(save_path):
        print(f"\nFound existing maps for {model_name} at {save_path}. Skipping generation.")
        continue

    print(f"\n{'='*60}\n  Computing attribution maps | Model: {model_name}\n{'='*60}")

    drive_weight_path = os.path.join(drive_model_dir, f"{model_name}_best_model.pth")
    model = mm.load_model(model_name, drive_weight_path, device)

    engine = att_engine.AttributionEngine(model, device)
    _engine_cache[id(model)] = engine

    n_already_done = len([
        f for f in os.listdir(_per_image_dir(SAVE_DIR, model_name))
        if f.endswith(".npz")
    ]) if os.path.isdir(_per_image_dir(SAVE_DIR, model_name)) else 0
    print(f"  {n_already_done}/{len(test_samples)} images already checkpointed for {model_name}; resuming...")

    run_attributions_on_samples(
        engine, test_samples, test_transform,
        save_dir=SAVE_DIR, model_name=model_name,
        desc=f"XAI maps ({model_name})",
    )

    attribution_results = assemble_attribution_results(SAVE_DIR, model_name, len(test_samples))

    gts           = np.stack(attribution_results['labels'], axis=0)
    gradcam_maps  = np.stack(attribution_results['gradcam'], axis=0)
    lime_maps     = np.stack(attribution_results['lime'], axis=0)
    shap_maps     = np.stack(attribution_results['shap'], axis=0)

    np.savez(save_path, gts=gts, gradcam=gradcam_maps, lime=lime_maps, shap=shap_maps)
    print(f"Saved {N_TEST} images' attribution maps to {save_path}")

    del attribution_results, gts, gradcam_maps, lime_maps, shap_maps

    mm.unload_model(model=model, engine=engine, engine_cache=_engine_cache)
    mm.gpu_mem_report(f"after unloading {model_name}")

In [ ]:
# with np.load(os.path.join(SAVE_DIR, "attribution_maps_resnet50_test.npz")) as data:
#     gts, gradcam_maps, lime_maps, shap_maps = data['gts'], data['gradcam'], data['lime'], data['shap']

# n = gts.shape[0]
# n_display = min(n, 5)

# raw_images = np.stack([test_samples[i][0] for i in range(n_display)], axis=0)  # only build what you'll show

# fig, axes = plt.subplots(n_display, 4, figsize=(16, 4 * n_display))
# if n_display == 1:
#     axes = axes[None, :]

# for i in range(n_display):
#     axes[i, 0].imshow(gts[i], cmap='gray')
#     axes[i, 0].set_title(f"Image {i}: Lesion Mask")
#     axes[i, 0].axis('off')

#     for j, (name, m) in enumerate([("GradCAM", gradcam_maps[i]),
#                                     ("LIME", lime_maps[i]),
#                                     ("SHAP", shap_maps[i])], start=1):
#         axes[i, j].imshow(raw_images[i])
#         axes[i, j].imshow(m, cmap='jet', alpha=0.5)
#         axes[i, j].set_title(name)
#         axes[i, j].axis('off')

# plt.tight_layout()
# plt.show()
# plt.close('all')

# del gts, gradcam_maps, lime_maps, shap_maps, raw_images
# mm.gc.collect()

# **Localization Evaluation: The pointing game and IoU**


**Pointing Game**: Takes the single highest-attribution pixel in the explanation map and checks if it falls inside the ground-truth lesion region. Scores 1 if yes, 0 if no. It's a strict, all-or-nothing test of whether the model's single most important pixel is disease-relevant.

**IoU**: Measures a model's localization performance by calculating the ratio of the overlap area to the combined total area of the predicted bounding box and the ground-truth bounding box


In [ ]:
def binarize_map_adaptive(attribution_map, gt_mask):
    """Schulz et al. (2020) style per-image adaptive thresholding."""
    m = np.asarray(attribution_map, dtype=np.float32)
    n = int(np.asarray(gt_mask).sum())

    if n <= 0:
        return np.zeros_like(m, dtype=np.uint8)
    if n >= m.size:
        return (m > 0).astype(np.uint8)

    top_k_frac = n / m.size
    cutoff = np.percentile(m, 100.0 * (1.0 - top_k_frac))
    return ((m >= cutoff) & (m > 0)).astype(np.uint8)

def binarize_map_batch(maps, gts):
    """Each map is thresholded using its own GT pixel count rather than a global threshold value."""
    assert maps.shape[0] == gts.shape[0]
    return np.stack([binarize_map_adaptive(maps[i], gts[i]) for i in range(maps.shape[0])], axis=0)

localization_rows = []

for model_name in MODEL_NAMES:
    load_path = os.path.join(SAVE_DIR, f"attribution_maps_{model_name}_test.npz")
    data = np.load(load_path)
    gts, gradcam_maps, lime_maps, shap_maps = data['gts'], data['gradcam'], data['lime'], data['shap']

    maps_dict = {"gradcam": gradcam_maps, "lime": lime_maps, "shap": shap_maps}

    print(f"=== {model_name} ===")
    for method, cont_maps in maps_dict.items():
        bin_maps = binarize_map_batch(cont_maps, gts)

        pointing_game = xb.hit(cont_maps, gts)
        iou_score = xb.iou(bin_maps, gts)

        print(f"--- {method} ---")
        print(f"Pointing Game           : {pointing_game:.4f}")
        print(f"IoU                     : {iou_score:.4f}\n")

        localization_rows.append({
            'model': model_name,
            'method': method,
            'pointing_game': round(pointing_game, 4),
            'iou': round(iou_score, 4),
        })

localization_df = pd.DataFrame(localization_rows)
display(localization_df)

In [ ]:
RESULTS_DIR = f"/data/{name}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
localization_df.to_csv(os.path.join(RESULTS_DIR, "localization_summary.csv"), index=False)
print(f"Saved localization summary to {os.path.join(RESULTS_DIR, 'localization_summary.csv')}")

In [ ]:
_trapz = getattr(np, "trapezoid", None) or np.trapz

In [ ]:
def load_attribution_maps(save_dir, model_name, filename_template="attribution_maps_{model}_test.npz"):
    """Loads precomputed attribution maps for one model from disk."""
    with np.load(os.path.join(save_dir, filename_template.format(model=model_name))) as data:
        method_maps = {
            "LIME": np.array(data['lime']),
            "GradCAM": np.array(data['gradcam']),
            "SHAP": np.array(data['shap']),
        }
    return method_maps

# **Faithfulness Evaluation: IROF**

Gradually remove the image regions identified as most important by the XAI method. If the model's confidence drops significantly, it shows those regions were genuinely important, meaning the explanation is more faithful.

**The original IROF paper uses SLIC to segment images into regions**

In [ ]:
class FastIROF:
    def __init__(self, model, device, n_segments=100, compactness=10, sigma=1, max_regions=100):
        """Initialize IROF evaluator with model and segmentation parameters"""
        self.model = model.to(device)
        self.device = device
        self.n_segments = n_segments    # SLIC: approximate target number of segments
        self.compactness = compactness  # SLIC: balances color proximity vs. spatial proximity
        self.sigma = sigma              # Gaussian smoothing applied before segmenting
        self.max_regions = max_regions  # Max number of regions to evaluate

        # ImageNet normalization stats used to invert test_transform before segmenting,
        # so SLIC sees real pixel colours/edges instead of normalized (mean-0) values.
        self._imagenet_mean = np.array([0.485, 0.456, 0.406])
        self._imagenet_std = np.array([0.229, 0.224, 0.225])

    def perturb(self, image, mask):
        """Replace selected region in the image with the image's average color (baseline)"""
        baseline = image.mean(dim=[2, 3], keepdim=True)
        image = image.clone()

        if mask.shape != (image.shape[2], image.shape[3]):
            raise ValueError(f"Mask shape {mask.shape} does not match image shape {(image.shape[2], image.shape[3])}")

        mask = mask.unsqueeze(0).unsqueeze(0)  # Add batch and channel dims
        mask = mask.expand_as(image)  # Match shape to image

        # Apply perturbation: replace masked area with baseline
        image = torch.where(mask.bool(), baseline, image)
        return image

    def segment_image(self, x):
        """Segment the original input image (not the attribution map)"""
        img_np = x.squeeze(0).permute(1, 2, 0).cpu().numpy()
        img_np = img_np * self._imagenet_std + self._imagenet_mean  # undo normalization
        img_np = np.clip(img_np, 0.0, 1.0)

        segments = slic(
            img_np.astype(np.float64),
            n_segments=self.n_segments,
            compactness=self.compactness,
            sigma=self.sigma,
            channel_axis=-1,   # img_np is now a real 3-channel RGB image, not a 1-channel map
            start_label=0,
        )
        return segments

    def apply(self, x, a, y, segments, batch_size=8, order="morf", seed=42):
        """order="morf" (Most-Relevant-First) removes regions in DECREASING order of attribution score
        order="random" removes regions in a fixed random order and is only meant to used as a baseline to compare against"""
        if order not in ("morf", "random"):
            raise ValueError(f"order must be 'morf' or 'random', got {order!r}")

        x = x.to(self.device)
        y = y.to(self.device)

        a = a.squeeze()
        if a.ndim == 3:
            a = a.mean(axis=0)

        if a.shape != (x.shape[2], x.shape[3]):
            raise ValueError(f"Attribution shape {a.shape} does not match image size {(x.shape[2], x.shape[3])}")
        if segments.shape != (x.shape[2], x.shape[3]):
            raise ValueError(f"Segments shape {segments.shape} does not match image size {(x.shape[2], x.shape[3])}")

        # Get original model score for the target class
        with torch.no_grad():
            output = self.model(x)
            orig_score = F.softmax(output, dim=1) * y  # Element-wise multiplication with target mask
            orig_score = orig_score.sum().item()

        # Compute average ATTRIBUTION value within each (fixed, image-derived) region --
        region_scores = [(seg_id, a[segments == seg_id].mean()) for seg_id in np.unique(segments)]

        if order == "morf":
            # Sort by attribution score, HIGHEST first -- remove the regions the
            # explanation thinks are most important first. This is what makes it IROF.
            region_scores.sort(key=lambda pair: pair[1], reverse=True)
        else:  # order == "random"
            rng = np.random.RandomState(seed)
            rng.shuffle(region_scores)

        # Optionally limit number of evaluated regions
        if len(region_scores) > self.max_regions:
            region_scores = region_scores[:self.max_regions]

        scores = [orig_score]
        cumulative_mask = np.zeros_like(segments, dtype=np.uint8)

        with torch.no_grad():
            batch_buffer = []
            for seg_id, _ in region_scores:
                cumulative_mask[segments == seg_id] = 1
                batch_buffer.append(self.perturb(x.clone(), torch.from_numpy(cumulative_mask).to(self.device)))

                if len(batch_buffer) == batch_size:
                    batch = torch.stack(batch_buffer).squeeze(1)
                    output = self.model(batch)
                    for out in output:
                        score = F.softmax(out, dim=0) * y.squeeze()
                        scores.append(score.sum().item())
                    batch_buffer = []
                    del batch, output

            if batch_buffer:  # leftover partial batch
                batch = torch.stack(batch_buffer).squeeze(1)
                output = self.model(batch)
                for out in output:
                    score = F.softmax(out, dim=0) * y.squeeze()
                    scores.append(score.sum().item())
                del batch, output

        eps = 1e-8
        normalized_scores = [s / (orig_score + eps) for s in scores]

        return scores, normalized_scores, segments


def resample_curve(curve, n_points=21):
    x_orig = np.linspace(0, 1, len(curve))
    x_new = np.linspace(0, 1, n_points)
    return np.interp(x_new, x_orig, curve)

In [ ]:
def run_irof_on_image(irof_evaluator, raw_img_np, predicted_class, method_maps, img_idx,
                       num_classes=5, transform=test_transform):
    """Runs MoRF IROF for one image, across every attribution method."""

    input_tensor = transform(PILImage.fromarray(raw_img_np)).unsqueeze(0).to("cpu")

    out_max = torch.zeros(1, num_classes)
    out_max[0, predicted_class] = 1

    image_segments = irof_evaluator.segment_image(input_tensor)# Segment the original image once, reuse across all methods

    rows = []
    sorted_curves = {}
    random_curves = {}

    for method_name, maps_array in method_maps.items():
        attr_map = torch.tensor(maps_array[img_idx], dtype=torch.float32)

        _, norm_scores_sorted, _ = irof_evaluator.apply(x=input_tensor, a=attr_map, y=out_max, segments=image_segments, order="morf")
        _, norm_scores_random, _ = irof_evaluator.apply(x=input_tensor, a=attr_map, y=out_max, segments=image_segments, order="random")

        resampled_sorted = resample_curve(norm_scores_sorted)
        resampled_random = resample_curve(norm_scores_random)

        sorted_curves[method_name] = resampled_sorted
        random_curves[method_name] = resampled_random
        
        irof_auc = float(_trapz(resampled_sorted) / (len(resampled_sorted) - 1))
        rows.append({
            'method':          method_name,
            'image_index':     img_idx,
            'predicted_class': predicted_class,
            'irof_auc':        round(irof_auc, 4),
            'num_regions':     len(np.unique(image_segments))
        })
        del attr_map

    del input_tensor, out_max, image_segments
    mm.clear_memory()
    
    return rows, sorted_curves, random_curves

In [ ]:
def run_irof_evaluation(model, samples, predicted_labels, method_maps, device,
                         n_segments=100, compactness=10, sigma=1, max_regions=100,
                         desc="IROF"):
    """Runs the full IROF faithfulness evaluation over samples."""


    irof_evaluator = FastIROF(model=model, device=device, n_segments=n_segments,
                               compactness=compactness, sigma=sigma, max_regions=max_regions)

    all_rows = []
    curves = {m: [] for m in method_maps}
    random_curves = {m: [] for m in method_maps}

    for img_idx in tqdm(range(len(samples)), desc=desc, ncols=90):
        raw_img_np, _ = samples[img_idx]
        predicted_class = int(predicted_labels[img_idx])

        rows, sorted_curves_i, random_curves_i = run_irof_on_image(
            irof_evaluator, raw_img_np, predicted_class, method_maps, img_idx
        )

        all_rows.extend(rows)
        for method_name in method_maps:
            curves[method_name].append(sorted_curves_i[method_name])
            random_curves[method_name].append(random_curves_i[method_name])

    full_df = pd.DataFrame(all_rows)
    return full_df, curves, random_curves

# **Why max_regions=100?**
Limiting to 100 regions ensures a good balance between spatial resolution and runtime efficiency. This number is sufficient to capture meaningful attribution patterns without over-segmenting the image or introducing noise from very small, low-relevance regions. At the same time, it limits the number of model evaluations during IROF scoring, keeping runtime manageable.

This choice follows common practice (e.g., skimage library).

In [ ]:
def summarize_and_save_irof(full_df, results_dir, tag="test"):
    """Aggregates per-image IROF results into a per-method summary."""

    summary = (full_df.groupby('method').agg(mean_irof_auc=('irof_auc', 'mean'), std_irof_auc=('irof_auc', 'std'),
             n_images=('irof_auc', 'count')).reset_index().sort_values('mean_irof_auc')
    )

    summary['mean_irof_auc'] = summary['mean_irof_auc'].round(4)
    summary['std_irof_auc'] = summary['std_irof_auc'].round(4)

    os.makedirs(results_dir, exist_ok=True)
    now = datetime.now().strftime("%Y-%m-%d")
    full_path = os.path.join(results_dir, f'irof_full_{tag}_{now}.csv')
    summary_path = os.path.join(results_dir, f'irof_summary_{tag}_{now}.csv')

    full_df.to_csv(full_path, index=False)
    summary.to_csv(summary_path, index=False)

    print(f"Saved per-image results to {full_path}")
    print(f"Saved summary to {summary_path}")

    return summary

In [ ]:
def plot_irof_curves(curves, n_images, title_suffix=""):
    """Plots the mean score-drop curve per method."""

    plt.figure(figsize=(10, 6))
    n_resample_points = len(next(iter(curves.values()))[0])
    fraction_removed = np.linspace(0, 1, n_resample_points)

    for method_name, method_curves in curves.items():
        mean_curve = np.array(method_curves).mean(axis=0)
        plt.plot(fraction_removed, mean_curve, marker="o", markersize=3, label=method_name)

    plt.title(f"IROF Score Drop Comparison{title_suffix} ({n_images} images)")
    plt.xlabel("Fraction of Regions Removed")
    plt.ylabel("Mean Normalized Prediction Score (1.0 = original confidence)")
    plt.grid(True)
    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
    plt.tight_layout()
    plt.show()
    plt.close() 


def plot_irof_bar(summary, title_suffix=""):
    """Plots a bar chart of mean IROF AUC per method."""
    plt.figure(figsize=(7, 5))
    plt.bar(summary['method'], summary['mean_irof_auc'])
    plt.ylabel("Mean IROF AUC (normalized, lower = more faithful)")
    plt.title(f"Faithfulness Comparison by Attribution Method{title_suffix}")
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close() 

**Running faithfulness test**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_trapz = getattr(np, "trapezoid", None) or np.trapz

all_irof_dfs = []
irof_curves_by_model = {}

for model_name in MODEL_NAMES:
    print(f"\n{'='*60}\n  Faithfulness (IROF) | Model: {model_name}\n{'='*60}")

    drive_weight_path = os.path.join(drive_model_dir, f"{model_name}_best_model.pth")
    model = mm.load_model(model_name, drive_weight_path, device)

    engine = att_engine.AttributionEngine(model, device)
    _engine_cache[id(model)] = engine

    method_maps = load_attribution_maps(SAVE_DIR, model_name)
    N_FAITHFULNESS_IMAGES = len(next(iter(method_maps.values())))
    faithfulness_samples = idrid_subset[:N_FAITHFULNESS_IMAGES]

    predicted_labels = get_predicted_labels(model, faithfulness_samples, device)

    print(f"Running faithfulness (IROF) test run on {N_FAITHFULNESS_IMAGES} images ")

    full_irof_df, irof_curves, irof_random_curves = run_irof_evaluation(
        model, faithfulness_samples, predicted_labels, method_maps, device=device
    )

    full_irof_df['model'] = model_name
    all_irof_dfs.append(full_irof_df)
    irof_curves_by_model[model_name] = irof_curves

    print(f"\nCollected {len(full_irof_df)} IROF measurements")

    irof_summary = summarize_and_save_irof(
        full_irof_df,
        results_dir=f"/data/{name}/results",
        tag=f"{model_name}_test"
    )

    print(f"\n=== Faithfulness (IROF) summary | {model_name} ===")
    print("    (lower mean_irof_auc = confidence drops faster once top-attributed regions")
    print("     are removed = more faithful explanation)\n")
    display(irof_summary)

    plot_irof_curves(irof_curves, N_FAITHFULNESS_IMAGES, title_suffix=f" ({model_name}, test run)")
    print()
    plot_irof_bar(irof_summary, title_suffix=f" ({model_name}, test run)")

    del method_maps, irof_random_curves
    mm.unload_model(model=model, engine=engine, engine_cache=_engine_cache)
    mm.gpu_mem_report(f"after unloading {model_name}")

full_irof_df_all_models = pd.concat(all_irof_dfs, ignore_index=True)
combined_irof_summary = (
    full_irof_df_all_models.groupby(['model', 'method'])
    .agg(mean_irof_auc=('irof_auc', 'mean'), std_irof_auc=('irof_auc', 'std'), n_images=('irof_auc', 'count'))
    .reset_index()
    .sort_values(['model', 'mean_irof_auc'])
)

combined_irof_summary['mean_irof_auc'] = combined_irof_summary['mean_irof_auc'].round(4)
combined_irof_summary['std_irof_auc'] = combined_irof_summary['std_irof_auc'].round(4)

print("\n=== Combined IROF summary (all models) ===")
display(combined_irof_summary)

# **Max-sensitivity**

In [ ]:
def explain_gradcam(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        img_tensor = torch.from_numpy(img).float()
        gradcam = eng.compute_gradcam(img_tensor.unsqueeze(0).to(eng.device), int(target))
        maps.append(gradcam)
        del img_tensor
    return np.stack(maps)[:, None, :, :]


def explain_lime(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        img_tensor = torch.from_numpy(img).float()
        raw = eng.unnormalize_to_uint8(img_tensor)
        explanation = eng.compute_lime(raw, int(target))
        heatmap = eng.get_lime_heatmap(explanation, int(target))
        maps.append(heatmap)
        eng.close_lime_explanation(explanation)
        del explanation, img_tensor, raw
    return np.stack(maps)[:, None, :, :]


def explain_shap(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        img_tensor = torch.from_numpy(img).float()
        raw = eng.unnormalize_to_uint8(img_tensor)
        shap_values = eng.compute_shap(raw, target_class=int(target), all_classes=False)
        heatmap, _ = eng.get_shap_heatmap(shap_values, class_position=0)
        maps.append(heatmap)
        del shap_values, img_tensor, raw
    return np.stack(maps)[:, None, :, :]

explain_funcs = {
    "GradCAM": explain_gradcam,
    "LIME": explain_lime,
    "SHAP": explain_shap,
}

In [ ]:
drive_model_dir = f"/data/{name}/models"
RESULTS_DIR = f"/data/{name}/results"
CHECKPOINT_DIR = f"/data/{name}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

**Max-sensitivity configuration**

In [ ]:
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "maxsens_per_image_checkpoint.csv")

CHECKPOINT_COLUMNS = ["model", "method", "image_index", "max_sensitivity"]


def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=CHECKPOINT_COLUMNS)


def append_checkpoint_row(row: dict):
    """Append a single completed (model, method, image_index) result to disk immediately."""
    df_row = pd.DataFrame([row], columns=CHECKPOINT_COLUMNS)
    write_header = not os.path.exists(CHECKPOINT_PATH)
    df_row.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)


def already_done(checkpoint_df, model_name, method, image_index):
    if checkpoint_df.empty:
        return False
    match = checkpoint_df[
        (checkpoint_df["model"] == model_name)
        & (checkpoint_df["method"] == method)
        & (checkpoint_df["image_index"] == image_index)
    ]
    return not match.empty


In [ ]:
MAXSENS_KWARGS = dict(
    nr_samples=10,
    lower_bound=0.2,
    norm_numerator=norm_func.fro_norm,
    norm_denominator=norm_func.fro_norm,
    similarity_func=similarity_func.difference,
)

METHOD_NAMES = ["GradCAM", "LIME", "SHAP"]  # processed strictly one at a time, in this order


**Running max-sensitivity**

In [ ]:

maxsens_samples = idrid_subset[:N_TEST]

checkpoint_df = load_checkpoint()
print(f"Resuming from checkpoint with {len(checkpoint_df)} previously completed (model, method, image) rows.")

# Fast O(1) membership check instead of re-scanning/rebuilding checkpoint_df every image.
completed_keys = set(zip(checkpoint_df["model"], checkpoint_df["method"], checkpoint_df["image_index"]))

for model_name in MODEL_NAMES:
    print(f"\n{'='*60}\n  MaxSensitivity | Model: {model_name}\n{'='*60}")

    # ---- Load ONE model at a time ----
    drive_weight_path = os.path.join(drive_model_dir, f"{model_name}_best_model.pth")
    model = mm.load_model(model_name, drive_weight_path, device)

    engine = att_engine.AttributionEngine(model, device)
    _engine_cache[id(model)] = engine

    # Build the normalized input batch 
    input_tensors = []
    for raw_img_np, _ in maxsens_samples:
        image_np = raw_img_np.astype(np.float32) / 255.0
        t = preprocess_image(image_np, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        input_tensors.append(t.squeeze(0))
    input_batch = torch.stack(input_tensors)
    x_batch_all = input_batch.cpu().numpy()
    del input_tensors, input_batch
    mm.clear_memory()

    y_batch_all = get_predicted_labels(model, maxsens_samples, device)
    mm.gpu_mem_report(f"after loading {model_name}")

    # ---- Process ONE XAI method at a time ----
    for method in METHOD_NAMES:
        explain_func = explain_funcs[method]
        print(f"\n  Method: {method} ({model_name})")

        metric = MaxSensitivity(**MAXSENS_KWARGS)

        n_images = x_batch_all.shape[0]
        pbar = tqdm(range(n_images), desc=f"MaxSensitivity [{model_name}/{method}]")

        for img_idx in pbar:
            if (model_name, method, img_idx) in completed_keys:
                continue  # already computed in a previous (crashed) run

            np.random.seed(SEED)
            random.seed(SEED)
            torch.manual_seed(SEED)

            x_single = x_batch_all[img_idx:img_idx + 1]
            y_single = y_batch_all[img_idx:img_idx + 1]
            a_single = None

            try:
                a_single = explain_func(model, x_single, y_single)

                score = metric(
                    model=model,
                    x_batch=x_single,
                    y_batch=y_single,
                    a_batch=a_single,
                    device=device,
                    explain_func=explain_func,
                )[0]

                row = {
                    "model": model_name,
                    "method": method,
                    "image_index": img_idx,
                    "max_sensitivity": float(score),
                }
                append_checkpoint_row(row)
                completed_keys.add((model_name, method, img_idx))

            except Exception as e:
                print(f"    Error on {model_name}/{method}/image {img_idx}: {e}")

            finally:
                # Clear Python/PyTorch memory after each image 
                a_single = None
                x_single = None
                y_single = None
                mm.clear_memory()
                if img_idx % 5 == 0:
                    mm.log_rss(f"{model_name}/{method}/image {img_idx}")

        # Clear the MaxSensitivity object after each  method
        del metric
        mm.clear_memory()

        checkpoint_df = load_checkpoint()
        method_df = checkpoint_df[
            (checkpoint_df["model"] == model_name) & (checkpoint_df["method"] == method)
        ]
        method_summary_path = os.path.join(
            RESULTS_DIR, f"maxsens_{model_name}_{method}_summary.csv"
        )
        method_df.to_csv(method_summary_path, index=False)
        print(f"  Saved {len(method_df)} results for {model_name}/{method} -> {method_summary_path}")
        mm.gpu_mem_report(f"after {model_name}/{method}")

    del x_batch_all, y_batch_all

    # Clear the model completely before loading the next one 
    mm.unload_model(model=model, engine=engine, engine_cache=_engine_cache)
    mm.gpu_mem_report(f"after unloading {model_name}")

print("\nAll models and methods complete.")


**Saving our max-sensitivity results**

In [ ]:
full_maxsens_df = load_checkpoint()
display(full_maxsens_df)

maxsens_summary = (
    full_maxsens_df
    .groupby(['model', 'method'])['max_sensitivity']
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .rename(columns={'mean': 'mean_max_sensitivity', 'std': 'std_max_sensitivity', 'count': 'n_images'})
)
maxsens_summary['mean_max_sensitivity'] = maxsens_summary['mean_max_sensitivity'].round(4)
maxsens_summary['std_max_sensitivity'] = maxsens_summary['std_max_sensitivity'].round(4)
maxsens_summary = maxsens_summary.sort_values(['model', 'mean_max_sensitivity'])

print("\n=== MaxSensitivity summary -- mean score per model/method ===")
print("    (lower mean_max_sensitivity = explanation changes less under small input")
print("     perturbations = more robust explanation)\n")
display(maxsens_summary)

now = datetime.now().strftime("%Y-%m-%d")
full_maxsens_df.to_csv(os.path.join(RESULTS_DIR, f'maxsens_full_{now}.csv'), index=False)
maxsens_summary.to_csv(os.path.join(RESULTS_DIR, f'maxsens_summary_{now}.csv'), index=False)
print(f"\nSaved per-image results to maxsens_full_{now}.csv")
print(f"Saved summary to maxsens_summary_{now}.csv")


**Visualizing our Max-Sensitivity results**

In [ ]:
plt.figure(figsize=(9, 5))
pivot = maxsens_summary.pivot(index='method', columns='model', values='mean_max_sensitivity')
pivot.plot(kind='bar', ax=plt.gca())
plt.yscale("log")
plt.ylabel("Mean MaxSensitivity (log scale, lower = more robust)")
plt.title("MaxSensitivity Comparison by Model and Method")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()